# Forschungsnahe Modelle mit HuggingFace Inference


Für diese Übung nutzen wir die **Hugging Face Inference API**, mit der Modelle direkt aus einem Notebook angesprochen werden können.

Über ein Notebook bzw. Code Modelle anzusprechen hat Vorteile gegenüber dem Aufruf über Chat Interfaces:
- Viele Prompts und Texte können nacheinander automatisch verarbeitet werden
- Ergebnisse lassen sich direkt weiterverarbeiten, visualisieren oder speichern
- Parameter - wie die Temperatur, welche bestimmt, wie deterministisch der Output eines Modells ist - können leicht angepasst werden

## Technische Voraussetzungen
Die folgenden Schritte sind in diesem Testrahmen optional, weil wir einen Token zur Verfügung stellen, werden aber nötig, wenn Ihr auch künftig mit Huggingface arbeiten möchtet.

- Benötigt wird ein kostenloser Huggingface Account (kann hier kreiert werden: https://huggingface.co/join)

- Ein Read Access Token zur Authentifizierung (Dafür diese Seite öffnen und via die Schaltfläche __create new token__ einen neuen Token mit der Berechtigung "READ" erstellen https://huggingface.co/settings/tokens)

## Weiterführende Ressourcen
Huggingface im GLAM-Bereich (Erklärung des HuggingFace-Hubs): https://huggingface.co/blog/hf-hub-glam-guide#what-is-the-hugging-face-hub


## Setup

Im folgenden Code greifen wir auf das Token für die Authentifizierung zu und legen das Modell fest, das wir nutzen möchten.

Probiert hier gerne verschiedene LLMs aus, z. B.:

`allenai/Olmo-3.1-32B-Instruct`
- 32B Modell entwickelt vom Allen Institute for AI mit starkem Fokus auf Transparenz und Reproduzierbarkeit

`meta-llama/Llama-3.3-70B-Instruct`
- leistungsstarkes und weit verbreitetes open weight 70B-Parameter LLM von Meta

`swiss-ai/Apertus-70B-Instruct-2509`
- 70B-Parameter Instruction-Modell aus einer europäischen KI-Initiative

Weitere über HF Inference verfügbare Modelle können hier gefunden werden: https://huggingface.co/models?inference_provider=all&sort=trending


In [ ]:
from huggingface_hub import InferenceClient
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

model = "allenai/Olmo-3.1-32B-Instruct"  # hier kann Modell verändert werden (allenai/Olmo-3.1-32B-Instruct; meta-llama/Llama-3.3-70B-Instruct; swiss-ai/Apertus-70B-Instruct-2509)

## Test: funktioniert alles?



In [ ]:
# User Prompt, System Prompt und zu verarbeitenden Text festlegen
user_prompt = "Please determine what the name of the cat is in the provided text."
system_prompt = "You always finish your responses saying meow"
text = "Bob is a cute cat."

In [ ]:
import time

# Funktion definieren
def testing_huggingface_api(text, user_prompt, system_prompt=None, model_name=model):
    """
    Testet ein Hugging Face Chat-Modell über die API.

    Args:
        text (str): Der Text, der an das Modell übergeben wird.
        user_prompt (str): Die Anweisung für das Modell.
        system_prompt (str, optional): Instruktion für das Systemverhalten.
        model_name (str): HuggingFace Modell, z. B. "allenai/Olmo-3.1-32B-Instruct".

    Returns:
        str: Die Antwort des Modells oder None bei Fehler.
    """
    print(f"Testing Hugging Face API :)")
    print(f"Text Length: {len(text)} characters")
    print(f"Using model: {model_name}")

    client = InferenceClient(api_key=HF_TOKEN)

    # System- und User-Messages zusammenbauen
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": f"{user_prompt}\n\n{text}"})

    # API Call
    print("Calling Inference API...")
    start_time = time.time()

    try:
        response = client.chat_completion(
            messages=messages,
            model=model_name,
            max_tokens=4000, # kann verändert werden
            temperature=0.0  # deterministisch - höhere Temperatur-Werte haben i. d. R. kreativere Antworten zur Folge
        )
        elapsed = time.time() - start_time
        print(f"API call completed in {elapsed:.2f} seconds")

        # Ergebnis extrahieren
        result = response.choices[0].message.content
        return result

    except Exception as e:
        print(f"Error: {e}")
        return None

# Aufruf der Funktion - hier übergeben wir unsere Prompts und den Text
result = testing_huggingface_api(
    text=text,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
    model_name="allenai/Olmo-3.1-32B-Instruct"
)

print(result)

Testing Hugging Face API :)
Text Length: 18 characters
Using model: allenai/Olmo-3.1-32B-Instruct
Calling Inference API...
API call completed in 1.41 seconds
The name of the cat is Bob. meow


## Mit API: Mehrere Anfragen automatisch verarbeiten
Wenn wir die API im Code nutzen, können wir mit einer for-Schleife mehrere Anfragen nacheinander absetzen.
So lassen sich Prompt und Modell automatisch auf viele Texte anwenden, zum Beispiel:

- auf alle Texte in einer Liste

- auf alle Einträge einer Spalte in einer CSV- oder Excel-Tabelle

Das spart Zeit und macht die Analyse großer Textmengen deutlich einfacher.

In [ ]:
texts = ["Bob is a cute cat", "I have a cat called Mimi", "My neighbour's black cat has no name"] # mehrere Texte in einer Liste
results = []

#iterativer Aufruf unserer Funktion
for text in texts:
    result = testing_huggingface_api(
        text=text,
        user_prompt=user_prompt,
        system_prompt=system_prompt,
        model_name="allenai/Olmo-3.1-32B-Instruct"
    )
    results.append(result)

# Ergebnis anschauen
for r in results:
    print(r)


Testing Hugging Face API :)
Text Length: 17 characters
Using model: allenai/Olmo-3.1-32B-Instruct
Calling Inference API...
API call completed in 1.45 seconds
Testing Hugging Face API :)
Text Length: 24 characters
Using model: allenai/Olmo-3.1-32B-Instruct
Calling Inference API...
API call completed in 1.89 seconds
Testing Hugging Face API :)
Text Length: 36 characters
Using model: allenai/Olmo-3.1-32B-Instruct
Calling Inference API...
API call completed in 2.19 seconds
The name of the cat is Bob. meow
The name of the cat is Mimi. meow
The cat in the provided text does not have a name. The text states, "My neighbour's black cat has no name." Therefore, the name of the cat is unknown or not given. meow


## Beispiel: Event Detection mit Large Language Models

Hier wird es schon recht komplex anhand eines realen geisteswissenschaftlichen Fallbeispiels. Wir versuchen aus historischen Zeitungsartikeln im JSON-Format Events zu extrahieren. Dabei wird der Prompt automatisiert nacheinander für die Felder einer Spalte in einer CSV-Datei angewendet und die Ausgaben des Modells in einer neuen Spalte "events" abgespeichert.

1. Nutzt den Prompt für die Erkennung narrativer Ereignisse und fügt diesen im Code ein.

2. Wendet verschiedene Modelle darauf an und vergleicht die Ergebnisse.

3. Vergleicht die Modell-Ausgaben mit den „idealen“ Ergebnissen.

**Reflexion:** Lässt sich das Ergebnis verbessern, wenn Ihr den Prompt oder die Modelleinstellungen anpasst?

In [ ]:
from huggingface_hub import InferenceClient
import json
import time
import pandas as pd

# CONFIGURATION
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
MODEL = "allenai/Olmo-3.1-32B-Instruct"  # You can change model here (allenai/Olmo-3.1-32B-Instruct; meta-llama/Llama-3.3-70B-Instruct; swiss-ai/Apertus-70B-Instruct-2509)

df = pd.read_csv("Earthquake_Articles.csv", sep=";")
column_name = "article_text" # this is the column in our dataframe that we are processing

# PROMPT
SYSTEM_PROMPT = """Test""" # copy the suggested prompt here

# ARTICLE
ARTICLE = """Add historical newspaper article here"""


# MAIN FUNCTION
def extract_events(article_text, model_name=MODEL):
    """
    Extract events from article using HF Inference API

    Args:
        article_text: The newspaper article text
        model_name: HuggingFace model to use

    Returns:
        Extracted events as string (JSON format)
    """
    print(f"Starting event extraction...")
    print(f"Article length: {len(article_text)} characters")
    print(f"Using model: {model_name}")

    # Initialize client
    client = InferenceClient(token=HF_TOKEN)

    # Prepare messages with proper system prompt
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"## EXTRACT FROM THIS TEXT:\n\n{article_text}\n\nReturn only JSON array."
        }
    ]

    # Call API
    print("Calling Inference API...")
    start_time = time.time()

    try:
        response = client.chat_completion(
            messages=messages,
            model=model_name,
            max_tokens=4000,
            temperature=0.0
        )

        elapsed = time.time() - start_time
        print(f"API call completed in {elapsed:.2f} seconds")

        # Extract result
        result = response.choices[0].message.content
        return result

    except Exception as e:
        print(f"Error: {e}")
        return None


def main():
    # Extract events
    df["events"] = None

    for i, row in df.iterrows():
        article_text = row[column_name]

        if pd.isna(article_text):
            continue

        events = extract_events(article_text)

        if events:
            print(events)

            # Try to parse and count
            try:
                # Clean potential markdown
                events_clean = events.replace("```json", "").replace("```", "").strip()
                events_list = json.loads(events_clean)
                df.at[i, "events"] = json.dumps(events_list)
            except:
                print("\nCould not parse as JSON, but extraction completed.")
                df.at[i, "events"] = events
        else:
            print("Extraction failed")

        time.sleep(2)

    df.to_csv("Earthquake_Articles_with_events.csv", index=False)


if __name__ == "__main__":
    main()

Starting event extraction...
Article length: 750 characters
Using model: allenai/Olmo-3.1-32B-Instruct
Calling Inference API...
API call completed in 4.95 seconds
[
  {
    "location": "Rome",
    "date": "January 21",
    "event": "deferment of insurance payment term for Reggio and Messina losses",
    "authority": "royal decree"
  },
  {
    "legal_change": "Death of an insured person now considered proved without further evidence (law of 12th inst.)"
  },
  {
    "insurance_simplification": "Production of policy not required if validity at catastrophe date can be established"
  },
  {
    "royal_response": "King Victor has issued a proclamation acknowledging public affection and devotion",
    "action": "Desires part of earthquake relief fund to be distributed immediately to those in pressing need"
  }
]
Starting event extraction...
Article length: 2442 characters
Using model: allenai/Olmo-3.1-32B-Instruct
Calling Inference API...
API call completed in 8.08 seconds
[
  {
    "locati

# Mit den eigenen Daten versuchen
Es kann helfen, erstmal den Test-Code hier rüber zu kopieren und dann zu versuchen diesen auf die eigenen Daten hin anzupassen, indem der übergebene Text und Prompt entsprechend der gewünschten Ergebnisse geändert wird. Oft sind auch LLMs leistungsstark darin, Code zu generieren und anzupassen.
Auch sind unten einige Codebeispiele für die verschiedenen Datentypen, die aufgeklappt, ausprobiert und angepasst werden können.

In [ ]:
# hier ist Platz für Code.

# CSV

### Testdatei erstellen

Das ist optional, um auszuprobieren, ob der Code funktioniert.

In [ ]:
import pandas as pd

# CSV-Datei
df = pd.DataFrame({
    "article_text": [
        "Bob ist eine süße Katze.",
        "Die meisten Katzen sind süß, aber manche sind auch gemein."
    ]
})
df.to_csv("test.csv", index=False, sep=";")

### Modell aufrufen

In [ ]:
from huggingface_hub import InferenceClient
import pandas as pd

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

MODEL = "allenai/Olmo-3.1-32B-Instruct"
SYSTEM_PROMPT = "Dein System-Prompt hier" # hier eigenen System-prompt formulieren
USER_PROMPT = "Bitte sag mir, was in der Datei steht." # hier eigenen User-Prompt formulieren

def call_model(text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{USER_PROMPT}\n\n{text}"}
    ]
    try:
        response = client.chat_completion(
            messages=messages,
            model=MODEL,
            max_tokens=4000,
            temperature=0.0
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return None

# --- CSV ---
df = pd.read_csv("test.csv", sep=";")
csv_outputs = []
max_texts = 10  # maximal 10 Texte verarbeiten
for i, text in enumerate(df["article_text"]):  # hier Spaltenname anpassen
    if i >= max_texts:
        break
    csv_outputs.append(call_model(text))
for text in df["article_text"]: # hier anstatt "article_text" Spaltenname aus der tatsächlichen Datei einfügen
    csv_outputs.append(call_model(text))

#Ergebnis anzeigen
print("CSV Outputs:", csv_outputs)

CSV Outputs: ['Natürlich! Hier ist der Inhalt der Datei:\n\n"Bob ist eine süße Katze."', 'Natürlich! Hier ist der Inhalt der Datei:\n\n"Die meisten Katzen sind süß, manche sind auch gemein."\n\nDas bedeutet, dass die meisten Katzen (Haustiere) freundlich oder liebenswürdig sind, aber es gibt auch Katzen, die lessen oder unbehaglicher sind.', 'Natürlich! Hier ist der Inhalt der Datei:\n\n"Bob ist eine süße Katze."', 'Natürlich! Hier ist der Inhalt der Datei:\n\n"Die meisten Katzen sind süß, manche sind auch gemein."\n\nDas bedeutet, dass die meisten Katzen (Haustiere) freundlich oder liebenswürdig sind, aber es gibt auch Katzen, die lessen oder unbehaglicher sind.']


# TXT

### Testdatei erstellen
Das ist optional, um auszuprobieren, ob der Code funktioniert.

In [ ]:
# TXT-Datei
txt_content = "Bob ist eine süße Katze.\nDie meisten Katzen sind süß, aber manche sind auch gemein."
with open("test.txt", "w", encoding="utf-8") as f:
    f.write(txt_content)

### Modell aufrufen

In [ ]:
from huggingface_hub import InferenceClient
import pandas as pd

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

MODEL = "allenai/Olmo-3.1-32B-Instruct" # Modell - kann geändert werden
SYSTEM_PROMPT = "Dein System-Prompt hier" # hier eigenen System-prompt formulieren
USER_PROMPT = "Bitte sag mir, was in der Datei steht." # hier eigenen User-Prompt formulieren

def call_model(text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{USER_PROMPT}\n\n{text}"}
    ]
    try:
        response = client.chat_completion(
            messages=messages,
            model=MODEL,
            max_tokens=4000,
            temperature=0.0
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return None

# --- TXT ---
with open("test.txt", "r", encoding="utf-8") as f: #hier eigenen Dateinamen einfügen
    txt_content = f.read()
txt_output = call_model(txt_content)

print("TXT Output:", txt_output)

#wenn mehrere txt-Dateien in einem Ordner verarbeitet werden sollen
"""
import glob
import time

txt_outputs = []
max_files = 10  # nur die ersten 10 Dateien verarbeiten

# Alle TXT-Dateien im Ordner "data" - Ordnername sollte angepasst werden
for i, path in enumerate(glob.glob("data/*.txt")):
    if i >= max_files:
        break
    with open(path, "r", encoding="utf-8") as f:
        txt_content = f.read()
    output = call_model(txt_content)
    txt_outputs.append({"file": path, "output": output})
    time.sleep(1)  # kleine Pause zwischen den Aufrufen

print(txt_outputs)
"""

TXT Output: Natürlich! Hier ist der Inhalt der Datei, den du mir gezeigt hast:

```
Bob ist eine süße Katze.
Die meisten Katzen sind süß, manche sind auch gemein.
```

Das bedeutet, dass die Datei zwei Sätze enthält:

1. „Bob ist eine süße Katze.“ – Das beschreibt Bob, der eine Katze ist und süß ist.
2. „Die meisten Katzen sind süß, manche sind auch gemein.“ – Das sagt aus, dass es unter Katzen meist süße gibt, aber auch manche, die gemein sind.

Wenn du noch weitere Informationen brauchst oder die Datei in einer bestimmten Form analysiert haben willst, sag gern Bescheid!


'\nimport glob\nimport time\n\ntxt_outputs = []\nmax_files = 10  # nur die ersten 10 Dateien verarbeiten\n\n# Alle TXT-Dateien im Ordner "data"\nfor i, path in enumerate(glob.glob("data/*.txt")):\n    if i >= max_files:\n        break\n    with open(path, "r", encoding="utf-8") as f:\n        txt_content = f.read()\n    output = call_model(txt_content)\n    txt_outputs.append({"file": path, "output": output})\n    time.sleep(1)  # kleine Pause zwischen den Aufrufen\n\nprint(txt_outputs)\n'

# XML

### Testdatei erstellen
Das ist optional, um auszuprobieren, ob der Code funktioniert.

In [ ]:
# XML-Datei
xml_content = """<?xml version="1.0" encoding="UTF-8"?>
<root>
    <article>Bob ist eine süße Katze.</article>
    <note>Die meisten Katzen sind süß, aber manche sind auch gemein.</note>
</root>
"""
with open("test.xml", "w", encoding="utf-8") as f:
    f.write(xml_content)

### Modell aufrufen

In [ ]:
from huggingface_hub import InferenceClient
import pandas as pd

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

MODEL = "allenai/Olmo-3.1-32B-Instruct" # Modell - kann geändert werden
SYSTEM_PROMPT = "Dein System-Prompt hier" # hier eigenen System-prompt formulieren
USER_PROMPT = "Bitte sag mir, was in der Datei steht." # hier eigenen User-Prompt formulieren

def call_model(text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{USER_PROMPT}\n\n{text}"}
    ]
    try:
        response = client.chat_completion(
            messages=messages,
            model=MODEL,
            max_tokens=4000,
            temperature=0.0
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return None

# --- XML ---
tree = ET.parse("test.xml")  # eigenen Dateinamen einfügen
root = tree.getroot()

# Variante 1: Alles auf einmal verarbeiten
all_text = " ".join([elem.text.strip() for elem in root.iter() if elem.text and elem.text.strip()])
result = call_model(all_text)
print("Ergebnis für das ganze XML:", result)

"""
# Variante 2: Tag-für-Tag
xml_outputs = []
for elem in root.iter():
    if elem.text and elem.text.strip():
        xml_outputs.append(call_model(elem.text.strip()))
print("Ergebnisse pro Tag:", xml_outputs)
"""

Ergebnis für das ganze XML: Natürlich! Hier ist der Inhalt der Datei, den du angehängt hast:

"Bob ist eine süße Katze. Die meisten Katzen sind süß, manche sind auch gemein."

Das bedeutet, dass die Datei eine kurze Beschreibung enthält, in der es um die Persönlichkeit von Bob, einer Katze, geht. Es wird gesagt, dass Bob süß ist, aber auch erwähnt, dass es unter Katzen sowohl süße als auch weniger angenehme Persönlichkeiten gibt.
Ergebnisse pro Tag: ['Natürlich! Hier ist der Inhalt der Datei:\n\n"Bob ist eine süße Katze."', 'Natürlich! Hier ist der Inhalt der Datei:\n\n"Die meisten Katzen sind süß, manche sind auch gemein."\n\nDas bedeutet, dass die meisten Katzen (Haustiere) freundlich oder liebenswürdig sind, aber es gibt auch Katzen, die lessen oder unbehaglicher sind.']


# PDF

### Testdatei erstellen

Das ist optional, um auszuprobieren, ob der Code funktioniert

In [ ]:
# PDF-Datei
!pip install fpdf
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.multi_cell(0, 10, "Bob ist eine süße Katze.\nDie meisten Katzen sind süß, manche sind auch gemein.")
pdf.output("test.pdf")

### Modell aufrufen

In [ ]:
!pip install pypdf2
from huggingface_hub import InferenceClient
import PyPDF2 # für PDFs
import time

# --------------------
# CONFIGURATION
# --------------------
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")  # HF-Token aus Secrets
MODEL = "allenai/Olmo-3.1-32B-Instruct"  # beliebiges Modell
SYSTEM_PROMPT = "Dein System-Prompt hier"
USER_PROMPT = "Dein User-Prompt hier"

client = InferenceClient(token=HF_TOKEN)

# --------------------
# Modellaufruf
# --------------------
def call_model(text, model_name=MODEL):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{USER_PROMPT}\n\n{text}"}
    ]
    try:
        response = client.chat_completion(
            messages=messages,
            model=model_name,
            max_tokens=4000,
            temperature=0.0
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return None
# --- PDF ---
pdf_file = open("test.pdf", "rb") # titel der Datei hier einfügen
reader = PyPDF2.PdfReader(pdf_file)
pdf_outputs = []
for page in reader.pages:
    text = page.extract_text()
    if text and text.strip():
        pdf_outputs.append(call_model(text.strip()))
pdf_file.close()

print("PDF Outputs:", pdf_outputs)

# Bonus: Spezialisierte Modelle

- Neben dem Zugriff auf LLMs über die API, gibt es auf Huggingface zahllose spezialisierte Modelle - auch für die geisteswissenschaftliche Forschung, z. B. für Aufgaben wie Sentimentanalyse or Named Entity Recognition. Schaut Euch gerne mal um und versucht, Modelle für spezifische Aufgaben zu finden und auszuprobieren: https://huggingface.co/models. Gefiltert werden kann zum Beispiel nach Aufgabe oder Sprachen, um passende Modelle zu finden.

- Wenn man eine spezielle Aufgabe erfüllt haben möchte, können diese Modelle - trotz kleinerer Größe - oft sogar bessere Ergebnisse erzielen als große, ressourcenintensivere General Purpose Modelle.